In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
sys.path.append("/home/cyf/wbi/Virginia/code")

import numpy as np
import torch
from importlib import reload

from CoarseFlow.datasets import synthetic_dataset
reload(synthetic_dataset)

from CoarseFlow.datasets.synthetic_dataset import (
    build_dataset_family,
    summarize_manifest,
    build_sameK_loader,
)

from wbi_0123.wholistic_registration.src.wholistic_registration.utils.simulation import generateMotion_Biophysical
from wbi_0123.wholistic_registration.src.wholistic_registration.utils import calFlow3d_Wei_v1
from wbi_0123.wholistic_registration.src.wholistic_registration.utils import IO

gut_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/raw/260201_test1_0_00002_TZCXY.ome.tif"
ventral_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/raw/260201_test1_0_00003_TZCXY.ome.tif"
dorsal_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/raw/260201_test1_0_00005_TZCXY.ome.tif"

gut_ref, gut_ref_desc = IO.readTiff(gut_Path)
ventral_ref, ventral_ref_desc = IO.readTiff(ventral_Path)
dorsal_ref, dorsal_ref_desc = IO.readTiff(dorsal_Path)

# 原始读入一般是 T/Z/C/X/Y 或类似格式；这里沿用你当前的写法
gut_volume1 = gut_ref.transpose(0, 2, 3, 1)[2, :, :, 60:80]
gut_volume2 = gut_ref.transpose(0, 2, 3, 1)[2, :, :, 80:100]
ventral_volume = ventral_ref.transpose(0, 2, 3, 1)[2, :, :, 50:70]
dorsal_volume = dorsal_ref.transpose(0, 2, 3, 1)[2, :, :, 60:90]
volumes = [
    {
        "name": "gut_60_80",
        "volume": gut_volume1,
        "ref_spacing": (1.0, 1.0, 3.0),
    },
    {
        "name": "gut_80_100",
        "volume": gut_volume2,
        "ref_spacing": (1.0, 1.0, 3.0),
    },
    {
        "name": "ventral_50_70",
        "volume": ventral_volume,
        "ref_spacing": (1.0, 1.0, 3.0),
    },
    {
        "name": "dorsal_60_90",
        "volume": dorsal_volume,
        "ref_spacing": (1.0, 1.0, 3.0),
    },
]

for i, v in enumerate(volumes):
    print(f"volume[{i}] shape:", v['volume'].shape, "dtype:", v['volume'].dtype, "min/max:", float(v['volume'].min()), float(v['volume'].max()))

CuPy is available with CUDA - using GPU acceleration
volume[0] shape: (1086, 538, 20) dtype: int16 min/max: -204.0 22528.0
volume[1] shape: (1086, 538, 20) dtype: int16 min/max: -215.0 21865.0
volume[2] shape: (1817, 630, 20) dtype: int16 min/max: -225.0 22746.0
volume[3] shape: (1701, 630, 30) dtype: int16 min/max: -358.0 25688.0


全局配置
---

In [ ]:
# build_dataset.ipynb - Cell 2

SAVE_ROOT = "/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6"

CONTROL_STRIDE = 16
CROP_SIZE_XY = (256, 256)

# 第一轮可以先用 DEBUG=True 快速测试流程；
# 确认没问题后改成 False 构建正式数据。
DEBUG = True

if DEBUG:
    N_STAGE0_TRAIN = 20
    N_STAGE0_VAL = 10

    N_STAGE1_TRAIN = 30
    N_STAGE1_VAL = 10

    N_STAGE2_TRAIN = 20
    N_STAGE2_VAL = 8

    N_STAGE3_TRAIN = 15
    N_STAGE3_VAL = 8
else:
    N_STAGE0_TRAIN = 200
    N_STAGE0_VAL = 80

    N_STAGE1_TRAIN = 300
    N_STAGE1_VAL = 80

    N_STAGE2_TRAIN = 120
    N_STAGE2_VAL = 50

    N_STAGE3_TRAIN = 80
    N_STAGE3_VAL = 40

print("SAVE_ROOT:", SAVE_ROOT)
print("DEBUG:", DEBUG)

SAVE_ROOT: /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_
DEBUG: True


Stage 0: sanity dataset
---

In [3]:
# Stage 0: sanity dataset

stage0_train_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage0_sanity_train"),
    split="stage0_train",

    K_values=(5,),
    sparse_steps=(5,),
    z_ratios=(3.0,),
    difficulties=("core",),

    num_samples_per_part=N_STAGE0_TRAIN,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0,),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=1000,
)

stage0_val_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage0_sanity_val"),
    split="stage0_val",

    K_values=(5,),
    sparse_steps=(5,),
    z_ratios=(3.0,),
    difficulties=("core",),

    num_samples_per_part=N_STAGE0_VAL,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0,),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=2000,
)

summarize_manifest(stage0_train_manifest)
summarize_manifest(stage0_val_manifest)

[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_train/stage0_train_K5_step5_zratioFallback3_core_N20.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_train/stage0_train_K5_step5_zratioFallback3_core_N20.json
[Saved manifest] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_train/stage0_train_manifest.json
[Total parts] 1
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_val/stage0_val_K5_step5_zratioFallback3_core_N10.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_val/stage0_val_K5_step5_zratioFallback3_core_N10.json
[Saved manifest] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_val/stage0_val_manifest.json
[Total parts] 1
Manifest: /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_train/stage

Stage 1:K=5 + variable spacing
---

In [4]:
# Stage 1: K=5, variable sparse_step and zRatio

stage1_train_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage1_K5_varSpacing_train"),
    split="stage1_train",

    K_values=(5,),
    sparse_steps=(3, 4, 5, 6, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("core",),

    num_samples_per_part=N_STAGE1_TRAIN,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.005, 0.01),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=3000,
)

stage1_val_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage1_K5_varSpacing_val"),
    split="stage1_val",

    K_values=(5,),
    sparse_steps=(3, 5, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("core",),

    num_samples_per_part=N_STAGE1_VAL,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.005, 0.01),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=4000,
)

summarize_manifest(stage1_train_manifest)
summarize_manifest(stage1_val_manifest)

[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback2_core_N30.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback2_core_N30.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback3_core_N30.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback3_core_N30.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback4_core_N30.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_K5_step3_zratioFallback4_core_N30.json
[Saved dataset] /home/cyf/wbi/Virg

Stage 2：variable K + variable spacing
---

In [5]:
# Stage 2: variable K, variable sparse_step, variable zRatio

stage2_train_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage2_varK_varSpacing_train"),
    split="stage2_train",

    K_values=(3, 4, 5, 6, 7),
    sparse_steps=(3, 4, 5, 6, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("core",),

    num_samples_per_part=N_STAGE2_TRAIN,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.005, 0.01, 0.02),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=5000,
)

stage2_val_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage2_varK_varSpacing_val"),
    split="stage2_val",

    K_values=(3, 5, 7),
    sparse_steps=(3, 5, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("core",),

    num_samples_per_part=N_STAGE2_VAL,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.01),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=6000,
)

summarize_manifest(stage2_train_manifest)
summarize_manifest(stage2_val_manifest)

[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback2_core_N20.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback2_core_N20.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback3_core_N20.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback3_core_N20.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback4_core_N20.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_K3_step3_zratioFallback4_core_N20.json
[Saved dataset] /home/

Stage 3：hard robustness dataset
---

In [6]:
# Stage 3: hard robustness dataset

stage3_train_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage3_hard_train"),
    split="stage3_train",

    K_values=(3, 5, 7),
    sparse_steps=(5, 6, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("hard",),

    num_samples_per_part=N_STAGE3_TRAIN,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.01, 0.02, 0.03),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=7000,
)

stage3_val_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "stage3_hard_val"),
    split="stage3_val",

    K_values=(3, 5, 7),
    sparse_steps=(5, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("hard",),

    num_samples_per_part=N_STAGE3_VAL,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.02, 0.03),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=8000,
)

summarize_manifest(stage3_train_manifest)
summarize_manifest(stage3_val_manifest)

[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback2_hard_N15.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback2_hard_N15.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback3_hard_N15.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback3_hard_N15.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback4_hard_N15.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_K3_step5_zratioFallback4_hard_N15.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage

Construct the eventual Datasets
---

In [7]:
# Final test dataset: broad but smaller

test_manifest = build_dataset_family(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,

    save_root=os.path.join(SAVE_ROOT, "final_test"),
    split="final_test",

    K_values=(3, 5, 7),
    sparse_steps=(3, 5, 8),
    z_ratios=(2.0, 3.0, 4.0),
    difficulties=("core", "hard"),

    num_samples_per_part=30 if not DEBUG else 5,
    crop_size_xy=CROP_SIZE_XY,
    noise_std_list=(0.0, 0.01, 0.03),

    control_stride=CONTROL_STRIDE,
    normalize=True,
    random_crop=True,
    random_z_start=True,
    return_motion=False,

    seed_base=9000,
)

summarize_manifest(test_manifest)

[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback2_core_N5.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback2_core_N5.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback2_hard_N5.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback2_hard_N5.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback3_core_N5.pt
[Saved meta   ] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback3_core_N5.json
[Saved dataset] /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/final_test/final_test_K3_step3_zratioFallback3_hard_N5.pt
[Saved 

Check the manifest
---

In [8]:
all_manifests = [
    stage0_train_manifest,
    stage0_val_manifest,
    stage1_train_manifest,
    stage1_val_manifest,
    stage2_train_manifest,
    stage2_val_manifest,
    stage3_train_manifest,
    stage3_val_manifest,
    test_manifest,
]

for p in all_manifests:
    print(p, "exists:", os.path.exists(p))

/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_train/stage0_train_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage0_sanity_val/stage0_val_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_train/stage1_train_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage1_K5_varSpacing_val/stage1_val_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_val/stage2_val_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_train/stage3_train_manifest.json exists: False
/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage3_hard_val/sta

Test 1 batch
---

In [10]:
# Check dataloader for Stage 2 variable-K dataset

loader, dataset, part_infos = build_sameK_loader(
    stage2_train_manifest,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

batch = next(iter(loader))

print("batch keys:", batch.keys())
print("mov:", batch["mov"].shape)          # (B,1,K,H,W)
print("ref:", batch["ref"].shape)          # (B,1,D,H,W)
print("spacing:", batch["spacing"].shape)  # (B,6)
print("z_init:", batch["z_init"].shape)    # (B,K)
print("gt_disp:", batch["gt_disp"].shape)  # (B,K,Hc,Wc,3)
print("gt_coords:", batch["gt_coords"].shape)

print("spacing[0]:", batch["spacing"][0])
print("z_init[0]:", batch["z_init"][0])

FileNotFoundError: [Errno 2] No such file or directory: '/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_/stage2_varK_varSpacing_train/stage2_train_manifest.json'

Summary the total manifest
---

In [ ]:
manifest_summary = {
    "stage0_train": stage0_train_manifest,
    "stage0_val": stage0_val_manifest,

    "stage1_train": stage1_train_manifest,
    "stage1_val": stage1_val_manifest,

    "stage2_train": stage2_train_manifest,
    "stage2_val": stage2_val_manifest,

    "stage3_train": stage3_train_manifest,
    "stage3_val": stage3_val_manifest,

    "final_test": test_manifest,
}

summary_path = os.path.join(SAVE_ROOT, "manifest_summary.json")

with open(summary_path, "w") as f:
    import json
    json.dump(manifest_summary, f, indent=2)

print("Saved manifest summary:", summary_path)

for k, v in manifest_summary.items():
    print(k, "->", v)